In [ ]:
from torch.utils.data import Dataset
import torch
import pandas as pd

class MnistTrainDataset(Dataset):
    def __init__(self, csv_file):
        # Load CSV (assumes first column is 'label', columns 1:784 are pixels)
        df = pd.read_csv(csv_file)
        
        # Extract labels and convert to PyTorch LongTensor
        self.labels = torch.tensor(df.iloc[:, 0].values, dtype=torch.long)
        
        # Extract pixels, normalize to [0, 1], reshape to (Channel, Height, Width)
        pixels = df.iloc[:, 1:].values / 255.0
        pixels = pixels.reshape(-1, 1, 28, 28) # 1 Channel, 28x28 pixels
        self.features = torch.tensor(pixels, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

In [ ]:
train_dataset = MnistTrainDataset("data/train.csv")

In [3]:
from torch import nn

class MNISTCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # 14x14
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)  # 7x7
        )
        self.classifier = nn.Linear(32 * 7 * 7, 10)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

In [11]:
def model_file(fold):
    return f"models/cnn_fold_{fold}.pth"

In [ ]:
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader, Subset
import numpy as np
from modules.earlystop import *

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
k_folds = 5
batch_size = 64
epochs = 100
skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)

verbose = 5
early_stop_patience=10

for fold, (train_idx, val_idx) in enumerate(skf.split(train_dataset.features, train_dataset.labels)):
    print(f"fold {fold+1}/{k_folds}")

    train_loader = DataLoader(Subset(train_dataset, train_idx), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(Subset(train_dataset, val_idx), batch_size=batch_size, shuffle=False)

    model = MNISTCNN().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    early_stop = EarlyStop(patience=early_stop_patience)
    best_val_loss = np.inf

    for epoch in range(epochs):
        model.train()

        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            optimizer.zero_grad()
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()

        model.eval()

        val_correct = val_loss = 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)

                logits = model(X_batch)
                loss = criterion(logits, y_batch)

                val_loss += loss.item() * X_batch.size(0)
                val_correct += (logits.argmax(1) == y_batch).sum().item()

        val_loss /= val_idx.shape[0]

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), model_file(fold+1))

        stop = early_stop(val_loss)

        if verbose and ((epoch+1)%verbose == 0 or epoch==0) or stop:
            print(f"epoch {epoch+1:02d} | val acc: {val_correct/len(val_idx):.5f}")

        if stop: break
    print(f"fold {fold+1} done")
print("done.")


fold 1/5
epoch 01 | val acc: 0.96619
epoch 05 | val acc: 0.98131
epoch 10 | val acc: 0.98238
epoch 15 | val acc: 0.98488
epoch 19 | val acc: 0.98476
fold 1 done
fold 2/5
epoch 01 | val acc: 0.96762
epoch 05 | val acc: 0.98000
epoch 10 | val acc: 0.98310
epoch 15 | val acc: 0.98690
epoch 18 | val acc: 0.98524
fold 2 done
fold 3/5
epoch 01 | val acc: 0.96369
epoch 05 | val acc: 0.98405
epoch 10 | val acc: 0.98464
epoch 15 | val acc: 0.98607
fold 3 done
fold 4/5
epoch 01 | val acc: 0.97060
epoch 05 | val acc: 0.98417
epoch 10 | val acc: 0.98500
epoch 15 | val acc: 0.98607
epoch 16 | val acc: 0.98536
fold 4 done
fold 5/5
epoch 01 | val acc: 0.96738
epoch 05 | val acc: 0.98024
epoch 10 | val acc: 0.98631
epoch 15 | val acc: 0.98619
epoch 18 | val acc: 0.98607
fold 5 done
done.


In [15]:
class MnistTestDataset(Dataset):
    def __init__(self, csv_file):
        # Load CSV (assumes first column is 'label', columns 1:784 are pixels)
        df = pd.read_csv(csv_file)
        
        # Extract pixels, normalize to [0, 1], reshape to (Channel, Height, Width)
        pixels = df.values / 255.0
        pixels = pixels.reshape(-1, 1, 28, 28) # 1 Channel, 28x28 pixels
        self.features = torch.tensor(pixels, dtype=torch.float32)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx]

In [16]:
test_dataset = MnistTestDataset("data/test.csv")
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [17]:
models = []
for fold in range(k_folds):
    m = MNISTCNN().to(device)
    m.load_state_dict(torch.load(model_file(fold+1)))
    m.eval()
    models.append(m)

predictions = []

with torch.no_grad():
    for X_batch in test_loader:
        X_batch = X_batch.to(device)

        logits_batch = torch.zeros(X_batch.size(0), 10).to(device)

        for m in models:
            logits_batch += torch.softmax(m(X_batch), dim=1)

        avg_logits = logits_batch / len(models)

        y_cap = avg_logits.argmax(dim=1)
        predictions.extend(y_cap.cpu().numpy())

In [18]:
res = pd.DataFrame({
    "ImageId": range(1, len(predictions)+1),
    "Label": predictions
})
res.to_csv("submissions/submission_pt.csv", index=False)